# Embedding Benchmark

This notebook demonstrates how to benchmark SNOMED CT embedding methods using the Hugging Face UMNSRS dataset for semantic similarity evaluation.

## Overview

The embedding benchmark evaluates how well vector embeddings capture semantic relationships between clinical concepts by comparing against human-annotated similarity scores from the [UMNSRS dataset](https://huggingface.co/datasets/bigbio/umnsrs) on Hugging Face Hub.

In [ ]:
%matplotlib inline

# Import embedding benchmark functions
from snomed_methods.benchmarking.embeddings import (
    evaluate_embedding_similarity,
)

# Import UMNSRS dataset loader
from snomed_methods.benchmarking.umnsrs import download_umnsrs

## Load the UMNSRS Dataset from Hugging Face

The UMNSRS dataset provides human-annotated similarity scores for SNOMED CT concept pairs.

In [ ]:
# Download and load the similarity subset
dataset = download_umnsrs("similarity")

print(f"UMNSRS Similarity Dataset: {len(dataset)} concept pairs")
pairs = list(dataset)[:5]
for i, pair in enumerate(pairs):
    print(f"  {i}: '{pair['text_1']}' vs '{pair['text_2']}'")
    print(f"     Score: {pair['label']}/1000 (>=500 = similar)")

## Load Your Embedding Method

Configure your ClinicalConceptEmbedder with the SapBERT model.

In [ ]:
# Load SapBERT from existing directory
from snomed_methods import ClinicalConceptEmbedder

embedder = ClinicalConceptEmbedder(
    model_name_or_path="/workspaces/snomed_methods/embedding_models/SapBERT-from-PubMedBERT-fulltext",
    backend="transformers",
    device="cpu",  # Change to 'cuda' if GPU is available
)

## Run the Embedding Benchmark

In [ ]:
# Evaluate using UMNSRS pairs
print("Running embedding benchmark...")

results = evaluate_embedding_similarity(
    embedder=embedder,
    dataset=list(dataset),  # All 566 pairs from UMNSRS similarity subset
    threshold=0.5,
    use_umnsrs_scores=True,
)

## Results

In [ ]:
# Display results
print("\n=== Embedding Benchmark Results ===")
for key, value in results.items():
    if isinstance(value, float):
        print(f"{key}: {value:.4f}")
    elif isinstance(value, int):
        print(f"{key}: {value}")

## Metrics Reference

| Metric | Description |
|--------|-------------|
| **Accuracy@Threshold** | Classification accuracy at similarity threshold |
| **Precision/Recall/F1** | Classification metrics |
| **Spearman Correlation** | Rank correlation with human scores |
| **Pearson Correlation** | Linear correlation with human scores |
| **MSE** | Mean squared error vs reference scores |